In [ ]:
# @title Загрузка, Декомпиляция и Сохранение { display-mode: "form" }
import os
import requests
import zipfile
import subprocess
from google.colab import files
from tqdm.notebook import tqdm
import shutil

URL = "https://originn0.github.io/dynamic_social_democracy/core.js" # @param {type:"string"}
core_js_path = 'core.js'
decompiler_js_path = 'decompiler.js'
output_dir = 'decompiled_output'
zip_output_path = 'source_backup.zip'

# 1. ЗАПИСЬ JAVASCRIPT КОДА
js_code = """
const fs = require('fs');
const path = require('path');
const vm = require('vm');

function extractJsonFromCoreJs(coreJsPath) {
    const content = fs.readFileSync(coreJsPath, 'utf8');
    const regex = /window\\.game\\s*=\\s*\\{\\s*(?:"compiled"|'compiled'|compiled)\\s*:\\s*/;
    const match = content.match(regex);
    if (!match) {
        try { return JSON.parse(content); } catch(e) {
            throw new Error("Could not find window.game={compiled: in " + coreJsPath);
        }
    }
    const afterMarker = content.substring(match.index + match[0].length).trim();
    const quoteChar = afterMarker[0];
    let jsonStrRaw = "";
    let escaped = false;
    let endIdx = -1;
    for (let i = 1; i < afterMarker.length; i++) {
        const c = afterMarker[i];
        if (escaped) { escaped = false; }
        else if (c === '\\\\\\\\') { escaped = true; }
        else if (c === quoteChar) { endIdx = i; break; }
    }
    if (endIdx === -1) throw new Error("Could not find end of compiled string");
    jsonStrRaw = afterMarker.substring(1, endIdx);
    const sandbox = { result: null };
    vm.createContext(sandbox);
    vm.runInContext(`result = ${quoteChar}${jsonStrRaw}${quoteChar}`, sandbox);
    try { return JSON.parse(sandbox.result); } catch (e) {
        throw new Error("Failed to parse game JSON: " + e.message);
    }
}

function camelToKebab(str) { return str.replace(/([a-z0-9])([A-Z])/g, '$1-$2').toLowerCase(); }

const scenePropertyMap = {
    onArrival: 'on-arrival', onDeparture: 'on-departure', onDisplay: 'on-display',
    viewIf: 'view-if', chooseIf: 'choose-if', maxVisits: 'max-visits',
    countVisitsMax: 'count-visits-max', maxVisitsVar: 'max-visits-var',
    newPage: 'new-page', setRoot: 'set-root', gameOver: 'game-over',
    isSpecial: 'is-special', setJump: 'set-jump', goTo: 'go-to',
    goSub: 'go-sub', goSubStart: 'go-sub-start', goSubEnd: 'go-sub-end',
    setBg: 'set-bg', setMusic: 'set-music', faceImage: 'face-image',
    wideImage: 'wide-image', bannerImage: 'banner-image', cardImage: 'card-image',
    isDeck: 'is-deck', isCard: 'is-card', isHand: 'is-hand', isPinnedCard: 'is-pinned-card',
    maxCards: 'max-cards', checkQuality: 'check-quality',
    broadDifficulty: 'broad-difficulty', narrowDifficulty: 'narrow-difficulty',
    difficultyScaler: 'difficulty-scaler', difficultyIncrement: 'difficulty-increment',
    checkSuccessGoTo: 'check-success-go-to', checkFailureGoTo: 'check-failure-go-to',
    minChoices: 'min-choices', maxChoices: 'max-choices', isTop: 'is-top',
    setSprites: 'set-sprites', setSpriteStyles: 'set-sprite-styles',
    setTopLeftStyle: 'set-top-left-style', setTopRightStyle: 'set-top-right-style',
    setBottomLeftStyle: 'set-bottom-left-style', setBottomRightStyle: 'set-bottom-right-style',
    unavailableSubtitle: 'unavailable-subtitle'
};

function decompileLogic(code, isPredicate = true) {
    if (typeof code !== 'string') return code;
    
    if (isPredicate) {
        let logic = code.startsWith('return ') ? code.substring(7).replace(/;$/, '') : code;
        logic = logic.replace(/\\\\(\\s*Q\\[['"]([^'"]+)['"]\\]\\s*\\|\\|\\s*0\\s*\\\\)/g, "$1");
        logic = logic.replace(/\\\\(\\s*Q\\.([a-zA-Z0-9_]+)\\s*\\|\\|\\s*0\\s*\\\\)/g, "$1");
        logic = logic.replace(/Q\\[['"]([^'"]+)['"]\\]/g, "$1");
        logic = logic.replace(/Q\\.([a-zA-Z0-9_]+)/g, "$1");
        logic = logic.replace(/state\\.visits\\[['"]([^'"]+)['"]\\]/g, '@$1');
        logic = logic.replace(/state\\.visits\\.([a-zA-Z0-9_]+)/g, '@$1');
        logic = logic.replace(/this\\.state\\.visits/g, "state.visits");
        logic = logic.replace(/\\\\(\\s*([a-zA-Z0-9_@]+)\\s*\\|\\|\\s*0\\s*\\\\)/g, "$1");
        logic = logic.replace(/\\\\(\\s*\\\\(\\s*([a-zA-Z0-9_@]+)\\s*\\\\)\\s*!==\\s*0\\s*\\\\)/g, '$1');
        logic = logic.replace(/\\\\(\\s*([a-zA-Z0-9_@]+)\\s*!==\\s*0\\s*\\\\)/g, '$1');
        logic = logic.replace(/\\s*!==\\s*0(?=[^0-9]|$)/g, '');
        logic = logic.replace(/===/g, ' = ').replace(/==/g, ' = ');
        logic = logic.replace(/!==/g, ' != ').replace(/!=/g, ' != ');
        logic = logic.replace(/&&/g, ' and ').replace(/\\|\\|/g, ' or ');
        logic = logic.replace(/!/g, 'not ');
        logic = logic.replace(/\\s+/g, ' ').trim();
        let changed = true;
        while (changed) {
            changed = false;
            if (logic.startsWith('(') && logic.endsWith(')')) {
                let count = 0, balanced = true;
                for (let i = 0; i < logic.length - 1; i++) {
                    if (logic[i] === '(') count++; if (logic[i] === ')') count--;
                    if (count === 0 && i > 0) { balanced = false; break; }
                }
                if (balanced) { logic = logic.substring(1, logic.length - 1); changed = true; }
            }
            let next = logic.replace(/\\\\(\\s*([a-zA-Z_@0-9.\\\\s'"]+(?:=|!=|<|>|<=|>=)[a-zA-Z_@0-9.\\\\s'"]+)\\\\)/g, "$1");
            if (next !== logic) { logic = next; changed = true; }
            next = logic.replace(/\\\\(\\s*([a-zA-Z_@0-9.]+)\\\\)/g, "$1");
            if (next !== logic) { logic = next; changed = true; }
            next = logic.replace(/\\\\(\\s*(not\\\\s+[a-zA-Z_@0-9.]+)\\\\)/g, "$1");
            if (next !== logic) { logic = next; changed = true; }
            next = logic.replace(/\\\\(\\s*(.*?)\\\\)\\s+(and|or)\\s+/g, "$1 $2 ");
            if (next !== logic) { logic = next; changed = true; }
            next = logic.replace(/\\s+(and|or)\\s+\\\\(\\s*(.*?)\\\\)/g, " $1 $2 ");
            if (next !== logic) { logic = next; changed = true; }
            logic = logic.replace(/\\s{2,}/g, ' ');
        }
        return logic.trim();
    } else {
        let actions = code;
        if (actions.includes('//') || actions.includes('/*') || actions.includes('if (') || actions.includes('if(') || actions.includes('{')) return actions;
        actions = actions.replace(/\\n/g, '; ');
        actions = actions.replace(/\\\\(\\s*Q\\[['"]([^'"]+)['"]\\]\\s*\\|\\|\\s*0\\s*\\\\)/g, "$1");
        actions = actions.replace(/\\\\(\\s*Q\\.([a-zA-Z0-9_]+)\\s*\\|\\|\\s*0\\s*\\\\)/g, "$1");
        actions = actions.replace(/Q\\[['"]([^'"]+)['"]\\]/g, "$1");
        actions = actions.replace(/Q\\.([a-zA-Z0-9_]+)/g, "$1");
        actions = actions.replace(/state\\.visits\\[['"]([^'"]+)['"]\\]/g, '@$1');
        actions = actions.replace(/state\\.visits\\.([a-zA-Z0-9_]+)/g, '@$1');
        actions = actions.replace(/([a-zA-Z_][a-zA-Z0-9_@]*)\\s*=\\s*\\1\\s*([\\+\\-\\*\\/])\\s*([^;]+)/g, '$1 $2= $3');
        actions = actions.replace(/([a-zA-Z_][a-zA-Z0-9_@]*)\\s*=\\s*([^;]+)/g, '$1 = $2');
        let lines = actions.split(';').map(s => s.trim()).filter(s => s);
        return lines.join('; ');
    }
}

function decompileContent(content, stateDependencies, isOneLine = false) {
    if (typeof content === 'string') return content;
    if (Array.isArray(content)) {
        let parts = content.map(c => decompileContent(c, stateDependencies, isOneLine));
        let isTopLevel = content.some(c => c && ['paragraph', 'heading', 'quotation', 'attribution', 'hrule'].includes(c.type));
        return (isTopLevel && !isOneLine) ? parts.join('\\n\\n') : parts.join('');
    }
    if (!content) return "";
    let localDeps = content.stateDependencies || stateDependencies;
    let result = "";
    if (content.type) {
        switch (content.type) {
            case 'paragraph': result = decompileContent(content.content, localDeps, isOneLine); break;
            case 'heading': result = "= " + decompileContent(content.content, localDeps, isOneLine); break;
            case 'quotation': result = "> " + decompileContent(content.content, localDeps, isOneLine); break;
            case 'attribution': result = ">> " + decompileContent(content.content, localDeps, isOneLine); break;
            case 'emphasis-1': result = "*" + decompileContent(content.content, localDeps, true) + "*"; break;
            case 'emphasis-2': result = "**" + decompileContent(content.content, localDeps, true) + "**"; break;
            case 'emphasis-3': result = "`" + decompileContent(content.content, localDeps, true) + "`"; break;
            case 'line-break': result = "//\\n"; break;
            case 'hrule': result = "---"; break;
            case 'conditional':
                let condText = "UNKNOWN";
                if (localDeps && localDeps[content.predicate] !== undefined) {
                     let dep = localDeps[content.predicate];
                     if (dep.fn && dep.fn.$code) condText = decompileLogic(dep.fn.$code);
                }
                result = `[? if ${condText} : ${decompileContent(content.content, localDeps, true)} ?]`;
                break;
            case 'insert':
                let insText = "UNKNOWN", qd = "";
                if (localDeps && localDeps[content.insert] !== undefined) {
                     let dep = localDeps[content.insert];
                     if (dep.fn && dep.fn.$code) insText = decompileLogic(dep.fn.$code);
                     if (dep.qdisplay) qd = " : " + dep.qdisplay;
                }
                result = `[+ ${insText}${qd} +]`;
                break;
            case 'magic': result = `{! ${content.content} !}`; break;
            case 'hidden': result = `[${decompileContent(content.content, localDeps, true)}]`; break;
            default: if (content.content) result = decompileContent(content.content, localDeps, isOneLine);
        }
    } else if (content.content) result = decompileContent(content.content, localDeps, isOneLine);
    return result;
}

function decompileScene(scene, rootId) {
    let lines = [];
    const isRoot = scene.id === rootId;
    if (!isRoot) {
        let shortId = scene.id;
        if (shortId.startsWith(rootId + ".")) shortId = shortId.substring(rootId.length + 1);
        lines.push(`@${shortId}`);
    }
    let stateDeps = scene.stateDependencies || (scene.content && scene.content.stateDependencies);
    const order = ['title', 'viewIf', 'chooseIf', 'onArrival', 'maxVisits', 'newPage', 'setRoot', 'goTo'];
    let keys = Object.keys(scene).filter(k => !['id', 'type', 'content', 'options', 'stateDependencies'].includes(k));
    keys.sort((a, b) => {
        let ka = order.indexOf(a), kb = order.indexOf(b);
        return (ka === -1 ? 999 : ka) - (kb === -1 ? 999 : kb);
    });
    for (let key of keys) {
        let value = scene[key];
        if (value === undefined || value === null) continue;
        if (key === 'countVisitsMax' && scene.maxVisits !== undefined && value === scene.maxVisits) continue;
        let dryKey = scenePropertyMap[key] || camelToKebab(key);
        if (['onArrival', 'onDeparture', 'onDisplay'].includes(key)) {
            let actions = value.map(a => a.$code).join('\\n').trim();
            let decompiled = decompileLogic(actions, false);
            if (decompiled.includes('\\n') || decompiled.includes('//') || decompiled.includes('/*') || decompiled.includes('{')) {
                lines.push(`${dryKey}: {!\\n${decompiled}\\n!}`);
            } else if (decompiled) lines.push(`${dryKey}: ${decompiled}`);
        } else if (['goTo', 'goSub', 'goSubStart', 'goSubEnd'].includes(key)) {
            if (Array.isArray(value)) {
                let parts = value.map(v => {
                    let sid = v.id;
                    if (sid.startsWith(rootId + ".")) sid = sid.substring(rootId.length + 1);
                    return sid + (v.predicate ? ` if ${decompileLogic(v.predicate.$code)}` : "");
                });
                lines.push(`${dryKey}: ${parts.join('; ')}`);
            } else {
                let sid = value;
                if (sid.startsWith(rootId + ".")) sid = sid.substring(rootId.length + 1);
                lines.push(`${dryKey}: ${sid}`);
            }
        } else if (key === 'tags') {
            lines.push(`${dryKey}: ${Array.isArray(value) ? value.join(', ') : value}`);
        } else if (value && value.$code) {
            lines.push(`${dryKey}: ${decompileLogic(value.$code)}`);
        } else if (typeof value === 'object' && !Array.isArray(value)) {
            lines.push(`${dryKey}: ${decompileContent(value, value.stateDependencies || stateDeps, true)}`);
        } else if (key === 'setSprites') {
            lines.push(`${dryKey}: ${Array.isArray(value) ? value.map(p => `${p[0]}: ${p[1]}`).join(', ') : value}`);
        } else lines.push(`${dryKey}: ${value}`);
    }
    if (scene.content) {
        let content = decompileContent(scene.content, stateDeps).trim();
        if (content) lines.push("", content);
    }
    if (scene.options && scene.options.length > 0) {
        lines.push("");
        scene.options.forEach(opt => {
            let shortId = opt.id;
            if (shortId.startsWith(rootId + ".")) shortId = shortId.substring(rootId.length + 1);
            if (!shortId.startsWith("@")) shortId = "@" + shortId;
            let line = `- ${shortId}`;
            if (opt.title) {
                let t = decompileContent(opt.title, stateDeps, true).trim();
                if (t) line += `: ${t}`;
            }
            lines.push(line);
            let optKeys = Object.keys(opt).filter(k => k !== 'id' && k !== 'title');
            optKeys.sort((a, b) => {
                let ka = order.indexOf(a), kb = order.indexOf(b);
                return (ka === -1 ? 999 : ka) - (kb === -1 ? 999 : kb);
            });
            for (let k of optKeys) {
                let val = opt[k];
                let dKey = scenePropertyMap[k] || camelToKebab(k);
                lines.push(`  ${dKey}: ${val && val.$code ? decompileLogic(val.$code) : val}`);
            }
        });
    }
    return lines.join('\\n');
}

function runDecompiler(game, outputDir, idToPathMap = {}) {
    if (!fs.existsSync(outputDir)) fs.mkdirSync(outputDir, { recursive: true });
    let info = "";
    const infoKeys = ['title', 'author', 'ifid'];
    infoKeys.forEach(k => { if (game[k]) info += `${k}: ${game[k]}\\n`; });
    for (let key in game) {
        if (['scenes', 'qualities', 'qdisplays', 'tagLookup', 'content', 'sections'].concat(infoKeys).includes(key)) continue;
        if (['string', 'number', 'boolean'].includes(typeof game[key])) info += `${camelToKebab(key)}: ${game[key]}\\n`;
    }
    fs.writeFileSync(path.join(outputDir, 'info.dry'), info);
    if (game.qdisplays) {
        const qdir = path.join(outputDir, 'qdisplays');
        if (!fs.existsSync(qdir)) fs.mkdirSync(qdir);
        for (let id in game.qdisplays) {
             let res = "\\n", qd = game.qdisplays[id];
             if (qd.content && Array.isArray(qd.content)) {
                 qd.content.forEach(range => {
                     let r = "(" + (range.min ?? "") + ".." + (range.max ?? "") + ") ";
                     r += decompileContent(range.output, null, true);
                     res += r + "\\n";
                 });
             }
             fs.writeFileSync(path.join(qdir, `${id}.qdisplay.dry`), res);
        }
    }
    if (game.qualities) {
        const qualDir = path.join(outputDir, 'qualities');
        if (!fs.existsSync(qualDir)) fs.mkdirSync(qualDir);
        for (let id in game.qualities) {
            let q = game.qualities[id], qlines = [];
            for (let key in q) if (key !== 'id') qlines.push(`${camelToKebab(key)}: ${q[key]}`);
            fs.writeFileSync(path.join(qualDir, `${id}.quality.dry`), qlines.join('\\n'));
        }
    }
    const scenesDir = path.join(outputDir, 'scenes');
    if (!fs.existsSync(scenesDir)) fs.mkdirSync(scenesDir);
    let rootScenes = {};
    const internalIds = ['prevScene', 'prevTopScene', 'jumpScene', 'backSpecialScene', 'returnScene'];
    for (let id in game.scenes) {
        if (internalIds.includes(id)) continue;
        let rootId = id.split('.')[0];
        if (!rootScenes[rootId]) rootScenes[rootId] = [];
        rootScenes[rootId].push(game.scenes[id]);
    }
    for (let rootId in rootScenes) {
        let scenes = rootScenes[rootId];
        let dryContent = "";
        scenes.forEach((scene, index) => {
            if (index > 0) dryContent += "\\n\\n";
            dryContent += decompileScene(scene, rootId);
        });
        let relPath = idToPathMap[rootId] || `${rootId}.scene.dry`;
        let filePath = path.join(scenesDir, relPath);
        if (relPath.startsWith('scenes/')) filePath = path.join(outputDir, relPath);
        const fileDir = path.dirname(filePath);
        if (!fs.existsSync(fileDir)) fs.mkdirSync(fileDir, { recursive: true });
        fs.writeFileSync(filePath, dryContent + "\\n");
    }
}

if (require.main === module) {
    const inputPath = process.argv[2], outputDir = process.argv[3] || 'decompiled', mapPath = process.argv[4];
    let idToPathMap = {};
    if (mapPath && fs.existsSync(mapPath)) idToPathMap = JSON.parse(fs.readFileSync(mapPath, 'utf8'));
    try {
        const game = extractJsonFromCoreJs(inputPath);
        runDecompiler(game, outputDir, idToPathMap);
        console.log(`Successfully decompiled ${inputPath} to ${outputDir}`);
    } catch (e) { console.error("Decompilation failed:", e.message); process.exit(1); }
}
"""

with open(decompiler_js_path, 'w', encoding='utf-8') as f:
    f.write(js_code)

# 2. ФУНКЦИИ ЗАГРУЗКИ И АРХИВАЦИИ
def download_with_progress(url, filename):
    print(f"🌐 Подключение к {url}...")
    response = requests.get(url, stream=True)
    response.raise_for_status()
    total_size = int(response.headers.get('content-length', 0))
    with open(filename, 'wb') as file, tqdm(
        desc="⬇️ Скачивание core.js",
        total=total_size, unit='iB', unit_scale=True, unit_divisor=1024,
    ) as bar:
        for data in response.iter_content(chunk_size=1024):
            size = file.write(data)
            bar.update(size)
    print(f"✅ Файл {filename} успешно скачан!\n")

def zip_directory(folder_path, zip_path):
    print(f"📦 Упаковка результатов в {zip_path}...")
    all_files = []
    for root, _, files_list in os.walk(folder_path):
        for file in files_list:
            all_files.append(os.path.join(root, file))
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file_path in tqdm(all_files, desc="🗜️ Архивирование файлов"):
            arcname = os.path.relpath(file_path, folder_path)
            zipf.write(file_path, arcname)
    print(f"✅ Архив создан!\n")

# ОСНОВНОЙ ПРОЦЕСС
if URL.strip():
    download_with_progress(URL, core_js_path)
else:
    print("⚠️ URL не указан. Загрузите core.js с компьютера:")
    uploaded = files.upload()
    if uploaded:
        core_js_path = list(uploaded.keys())[0]
        print(f"✅ Файл {core_js_path} загружен!\n")
    else:
        print("❌ Файл не был загружен.")

if os.path.exists(core_js_path):
    print(f"🔍 Файл core.js готов (Размер: {os.path.getsize(core_js_path)} байт).")
    print("⚙️ Запуск JS декомпилятора через Node.js...")
    result = subprocess.run(
        ['node', decompiler_js_path, core_js_path, output_dir],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        print("❌ ОШИБКА ДЕКОМПИЛЯЦИИ:")
        print(result.stderr)
    else:
        print("✅", result.stdout.strip())
        if os.path.exists(output_dir):
            zip_directory(output_dir, zip_output_path)
            if os.path.exists(zip_output_path):
                print("📥 Подготовка к скачиванию архива на ваше устройство...")
                files.download(zip_output_path)
            else:
                print("❌ Ошибка при создании архива.")
        else:
            print("❌ Ошибка: Папка с результатами не была создана декомпилятором.")
else:
    print("❌ Ошибка: Файл core.js не найден. Процесс остановлен.")
